# 05 – Multi-Run Comparison

Side-by-side comparison of all OIFS SST-sensitivity experiments for Hurricane Kirk:
- **baserun**: Control (observed SST)
- **+3K**: SST +3 K warming
- **+6K**: SST +6 K warming

Panels:
1. Storm track map (all runs + IBTrACS)
2. MSLP minimum time series
3. Cyclone Phase Space overlay
4. R17/R34 wind radius time series

In [ ]:
import sys, os
sys.path.insert(0, '../scripts')
sys.path.insert(0, '../ribberink_code')

import numpy as np
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print('cartopy not installed – map plot will use plain axes')

from oifs_adapter import RUNS, OIFSRun

## 1. Load pre-computed tracks and CPS data

In [ ]:
tracks = {}
for run_name in RUNS:
    safe_name = run_name.replace('+', 'p')
    path = f'../plots/tracks/track_{safe_name}.nc'
    if os.path.exists(path):
        tracks[run_name] = xr.open_dataset(path)
        print(f'Track loaded: {run_name} ({len(tracks[run_name].time)} steps)')

cps = {}
for run_name in RUNS:
    safe_name = run_name.replace('+', 'p')
    path = f'../plots/cps/cps_{safe_name}.nc'
    if os.path.exists(path):
        cps[run_name] = xr.open_dataset(path)
        print(f'CPS loaded: {run_name}')

In [ ]:
# Load IBTrACS best track if available
ibtracs_path = '../data/IBTrACS.NA.v04r00.nc'
obs_track = None
if os.path.exists(ibtracs_path):
    import hurricane_functions as hf
    obs_track = hf.import_ibtracs(hurr_year=2024, storm_name=b'KIRK')
    print(f'IBTrACS Kirk loaded: {len(obs_track)} records')

## 2. Storm Track Map

In [ ]:
colors = {'baserun': '#1f77b4', '+3K': '#d62728', '+6K': '#ff7f0e', '-3K': '#2ca02c'}

if HAS_CARTOPY:
    fig = plt.figure(figsize=(12, 7))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#d9d9d9')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.gridlines(draw_labels=True, linewidth=0.3)
    ax.set_extent([-90, -5, 5, 60], crs=ccrs.PlateCarree())
    transform = ccrs.PlateCarree()
else:
    fig, ax = plt.subplots(figsize=(12, 7))
    transform = None

for run_name, track in tracks.items():
    kw = dict(color=colors.get(run_name, 'k'), lw=2, label=run_name)
    if transform:
        ax.plot(track.lon, track.lat, transform=transform, **kw)
    else:
        ax.plot(track.lon, track.lat, **kw)

if obs_track is not None:
    kw = dict(color='k', lw=2, ls='--', label='IBTrACS')
    if transform:
        ax.plot(obs_track['lon'], obs_track['lat'], transform=transform, **kw)
    else:
        ax.plot(obs_track['lon'], obs_track['lat'], **kw)

ax.set_title('Hurricane Kirk 2024 – Storm Tracks (OIFS + IBTrACS)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('../plots/kirk_track_comparison.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_track_comparison.png')

## 3. MSLP Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for run_name, track in tracks.items():
    ax.plot(track.time, track.msl / 100, '.-', color=colors.get(run_name, 'k'),
            label=run_name)

if obs_track is not None and 'pres' in obs_track.columns:
    ax.plot(obs_track['datetime'], obs_track['pres'].astype(float),
            'k--', label='IBTrACS (obs)')

ax.set_xlabel('Date')
ax.set_ylabel('MSLP (hPa)')
ax.set_title('Hurricane Kirk 2024 – Minimum Sea-Level Pressure')
ax.invert_yaxis()
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout()
plt.savefig('../plots/kirk_mslp_comparison.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_mslp_comparison.png')

## 4. Cyclone Phase Space Overlay

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
panel_titles = ['Upper CPS (300-600 hPa)', 'Lower CPS (600-900 hPa)']

for ax, (key_sym, key_th), title in zip(
    axes,
    [('B', 'VT_upper'), ('B', 'VT_lower')],
    panel_titles
):
    for run_name, cps_ds in cps.items():
        if key_sym in cps_ds and key_th in cps_ds:
            B = cps_ds[key_sym].values
            VT = cps_ds[key_th].values
            ax.plot(B, VT, '.-', color=colors.get(run_name, 'k'), label=run_name)
            ax.plot(B[0], VT[0], 'o', ms=8, color=colors.get(run_name, 'k'))
            ax.plot(B[-1], VT[-1], 's', ms=8, color=colors.get(run_name, 'k'))

    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(0, color='k', lw=0.8)
    ax.set_xlabel('B (Thermal asymmetry, m)')
    ax.set_ylabel('$-V_T$ (m)')
    ax.set_title(title)
    ax.text(+20, +50, 'Warm\nSymmetric', ha='center', va='center', fontsize=9, color='#555')
    ax.text(-20, +50, 'Warm\nAsymmetric', ha='center', va='center', fontsize=9, color='#555')
    ax.text(+20, -50, 'Cold\nSymmetric', ha='center', va='center', fontsize=9, color='#555')
    ax.text(-20, -50, 'Cold\nAsymmetric', ha='center', va='center', fontsize=9, color='#555')

axes[0].legend(loc='upper right')
fig.suptitle('Hurricane Kirk 2024 – Cyclone Phase Space (Hart 2003)', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/kirk_cps_overlay.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_cps_overlay.png')

## 5. Summary statistics

In [ ]:
import pandas as pd

rows = []
for run_name, track in tracks.items():
    mslp_min = float(track.msl.min()) / 100  # hPa
    lat_at_min = float(track.lat.isel(time=int(track.msl.argmin())))
    rows.append({
        'Run': run_name,
        'Min MSLP (hPa)': f'{mslp_min:.1f}',
        'Lat at min': f'{lat_at_min:.1f}°N',
        'Track length (steps)': len(track.time)
    })

df = pd.DataFrame(rows).set_index('Run')
print(df.to_string())
df.to_csv('../plots/kirk_summary_stats.csv')
print('Saved: ../plots/kirk_summary_stats.csv')